# Task 1

In [1]:
import pandas as pd
import numpy as np
import random
import pyarrow as pa
import pyarrow.parquet as pq
cities = ["Berlin", "Seoul", "Nairobi", "Toronto", "Lima", "Tokyo", "Paris", "New York", "Sydney", "Cape Town"]
np.random.seed(42)
n = 500000
df = pd.DataFrame(
    {'user_id': np.arange(1, n+1),
     'city': np.random.choice(cities, n),
     'score': np.random.uniform(0,100, n),
     'active': np.random.choice([True,False], n),
     'signup_date': pd.to_datetime('today') - pd.to_timedelta(np.random.randint(0, 1095, n), unit='D') , 
     'age': np.random.randint(18,81,n),
     'sessions' : np.random.randint(0,501,n),
     'revenue' : np.random.uniform(0,1001,n)
    }
)
df

,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,Paris,43.689490,False,2026-02-01 16:47:58.714060,71,323,179.344026
1,2,Toronto,97.403462,True,2025-06-02 16:47:58.714060,39,352,615.508750
2,3,New York,66.148238,False,2025-11-08 16:47:58.714060,23,279,217.286869
3,4,Lima,21.993544,False,2025-05-22 16:47:58.714060,20,129,195.654732
4,5,Paris,91.719953,True,2023-04-02 16:47:58.714060,67,199,617.966939
...,...,...,...,...,...,...,...,...
499995,499996,Toronto,61.531318,False,2025-04-19 16:47:58.714060,67,448,157.600037
499996,499997,Berlin,74.712924,False,2025-10-25 16:47:58.714060,74,36,418.537187
499997,499998,Sydney,48.430697,True,2023-09-03 16:47:58.714060,27,205,803.925191
499998,499999,New York,89.680757,True,2026-01-24 16:47:58.714060,50,432,602.652148


In [2]:
df.to_parquet("user_new.parquet", index=False)
 
parquet_file = pq.ParquetFile("user_new.parquet")
print(f"Number of row groups: {parquet_file.metadata.num_row_groups}")
print(f"Number of columns: {parquet_file.metadata.num_columns}")
print(f"Number of rows: {parquet_file.metadata.num_rows}")
row_group = parquet_file.metadata.row_group(0)

for i in range(row_group.num_columns):
    col = row_group.column(i)

    print("Column:", col.path_in_schema)
    print("Type:", col.physical_type)
    print("Compressed size:", col.total_compressed_size)

    stats = col.statistics
    if stats is not None:
        print("Min:", stats.min)
        print("Max:", stats.max)
        print("Nulls:", stats.null_count)

    print()

Number of row groups: 1
Number of columns: 8
Number of rows: 500000
Column: user_id
Type: INT32
Compressed size: 2590600
Min: 1
Max: 500000
Nulls: 0

Column: city
Type: BYTE_ARRAY
Compressed size: 251179
Min: Berlin
Max: Toronto
Nulls: 0

Column: score
Type: DOUBLE
Compressed size: 4279334
Min: 5.188445665327279e-05
Max: 99.99983148609545
Nulls: 0

Column: active
Type: BOOLEAN
Compressed size: 62553
Min: False
Max: True
Nulls: 0

Column: signup_date
Type: INT64
Compressed size: 697342
Min: 2023-03-07 16:47:58.714060
Max: 2026-03-05 16:47:58.714060
Nulls: 0

Column: age
Type: INT32
Compressed size: 376346
Min: 18
Max: 80
Nulls: 0

Column: sessions
Type: INT32
Compressed size: 565610
Min: 0
Max: 500
Nulls: 0

Column: revenue
Type: DOUBLE
Compressed size: 4279334
Min: 0.0009968016080641462
Max: 1000.9978847998774
Nulls: 0



In [3]:
df.to_csv('users.csv', index=False)

In [4]:
import os
csv_size = os.path.getsize('users.csv')
csv_size/1024

44240.626953125

In [5]:
pq_size = os.path.getsize('user_new.parquet')
pq_size/1024

12800.0673828125

In [6]:
print(csv_size/pq_size)

3.4562807858753795


Parquet stores data in a columnar format, while CSV stores data as plain text rows. 
Parquet metadata contains important information such as:
- Number of row groups
- Column data types
- Column statistics (min, max, null count)
- Compressed sizes
CSV files do not contain any metadata. When reading a CSV file, the system must scan the entire file and infer column types.

# Task 2

In [7]:
%timeit pd.read_parquet("user_new.parquet")

132 ms ± 7.72 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
%timeit pd.read_parquet("user_new.parquet" , columns = ['city','score'])

128 ms ± 8.4 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [9]:
%timeit pd.read_csv("users.csv" , usecols = ['city','score'])

721 ms ± 12.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Parquet selective reads are faster because Parquet stores data in a columnar format.
In Parquet files, each column is stored separately in column chunks inside row groups.
This means that when a query requests only specific columns, the system reads only the required column chunks from disk.
For example, when we read only the 'city' and 'score' columns, Parquet loads only those two columns instead of the entire dataset.

# Task 3

In [10]:
%timeit pq.read_table('users.parquet', filters=[('age', '>', 50)])

9.55 s ± 354 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
%timeit df = pd.read_parquet('users.parquet'); df[df['age'] > 50]

24.4 s ± 2.36 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


- for Step 1 => 10.6 s ± 689 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
- for Step 2 => 25 s ± 1.7 s per loop (mean ± std. dev. of 7 runs, 1 loop each)
The performance gap exists because Predicate Pushdown filters the data at the storage level, reading only the necessary rows into memory.
Conversely, the Pandas approach loads the entire dataset into RAM first, which increases I/O overhead and memory consumption before any 
filtering occurs.

In [12]:
import duckdb
%timeit duckdb.sql("SELECT * FROM 'user_new.parquet' WHERE age > 50").df()


127 ms ± 3.26 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


- for Step 1 => 10.6 s ± 689 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
- for Step 2 => 25 s ± 1.7 s per loop (mean ± std. dev. of 7 runs, 1 loop each)
- for Step 4 => 94.5 ms ± 3.56 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)

# Task 4

In [13]:
df_r = df.groupby('city').size()
print(df_r)
%timeit df.groupby('city').size()

city
Berlin       50059
Cape Town    50061
Lima         50216
Nairobi      50113
New York     49885
Paris        49982
Seoul        49970
Sydney       49724
Tokyo        50059
Toronto      49931
dtype: int64
69.4 ms ± 2.51 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [14]:
records = duckdb.sql("SELECT city, count(*) as records_per_city from df group by city").df()
print(records)
%timeit duckdb.sql("SELECT city, count(*) as records_per_city from df group by city").df()

        city  records_per_city
0     Berlin             50059
1       Lima             50216
2  Cape Town             50061
3     Sydney             49724
4    Nairobi             50113
5      Seoul             49970
6      Tokyo             50059
7    Toronto             49931
8   New York             49885
9      Paris             49982
28.5 ms ± 2.19 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [15]:
avg_score_cit_df = df.groupby('city')['score'].mean().sort_values(ascending = False)
print(avg_score_cit_df)
%timeit df.groupby('city')['score'].mean().sort_values(ascending = False)

city
Tokyo        50.311448
Lima         50.151899
Seoul        50.135854
Paris        50.075511
New York     50.063780
Berlin       50.013372
Toronto      50.009568
Sydney       49.996347
Cape Town    49.994487
Nairobi      49.879502
Name: score, dtype: float64
69 ms ± 1.28 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [16]:
avg_score_city = duckdb.sql("SELECT city,avg(score) from df group by city order by avg(score) desc")
print(avg_score_city)
%timeit duckdb.sql("SELECT city,avg(score) from df group by city order by avg(score) desc")

┌───────────┬────────────────────┐
│   city    │     avg(score)     │
│  varchar  │       double       │
├───────────┼────────────────────┤
│ Tokyo     │ 50.311447639720384 │
│ Lima      │  50.15189868104277 │
│ Seoul     │  50.13585407261415 │
│ Paris     │  50.07551149964857 │
│ New York  │  50.06377999045983 │
│ Berlin    │  50.01337161408978 │
│ Toronto   │  50.00956774673048 │
│ Sydney    │ 49.996347191623634 │
│ Cape Town │ 49.994487181000736 │
│ Nairobi   │  49.87950214845574 │
├───────────┴────────────────────┤
│ 10 rows              2 columns │
└────────────────────────────────┘

3.35 ms ± 150 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [17]:
target_condition = (df['active'] == True) & (df['score'] > 75)
pandas_res = (df.groupby('city').apply(lambda x: (target_condition[x.index].sum() / len(x)) * 100, include_groups=False)).round(1)
print(pandas_res)
%timeit (df.groupby('city').apply(lambda x: (target_condition[x.index].sum() / len(x)) * 100, include_groups=False)).round(1)

city
Berlin       12.5
Cape Town    12.5
Lima         12.5
Nairobi      12.3
New York     12.5
Paris        12.6
Seoul        12.7
Sydney       12.5
Tokyo        12.7
Toronto      12.7
dtype: float64
159 ms ± 652 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [18]:
perc_active = duckdb.sql('''SELECT city, ROUND(SUM(CASE WHEN active = True and score >75  THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) 
as active_pct from df group by city ''')
print(perc_active)
%timeit duckdb.sql('''SELECT city, ROUND(SUM(CASE WHEN active = True and score >75  THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)  as active_pct from df group by city''')

┌───────────┬────────────┐
│   city    │ active_pct │
│  varchar  │   double   │
├───────────┼────────────┤
│ New York  │       12.5 │
│ Tokyo     │       12.7 │
│ Seoul     │       12.7 │
│ Paris     │       12.6 │
│ Toronto   │       12.7 │
│ Nairobi   │       12.3 │
│ Cape Town │       12.5 │
│ Sydney    │       12.5 │
│ Lima      │       12.5 │
│ Berlin    │       12.5 │
├───────────┴────────────┤
│ 10 rows      2 columns │
└────────────────────────┘

3.62 ms ± 62.1 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [19]:
top_10_user = df.assign(rank=df.groupby('city')['score'].rank(method='first', ascending=False)).query('rank <= 10')
print(top_10_user)
%timeit df.assign(rank=df.groupby('city')['score'].rank(method='first', ascending=False)) .query('rank <= 10')

        user_id      city      score  active                signup_date  age  \
2438       2439     Paris  99.995192    True 2024-03-09 16:47:58.714060   55   
5789       5790  New York  99.977347    True 2024-06-28 16:47:58.714060   44   
7100       7101   Toronto  99.998401   False 2023-07-22 16:47:58.714060   33   
8135       8136    Berlin  99.977353   False 2024-06-03 16:47:58.714060   41   
12513     12514    Sydney  99.986537    True 2023-12-30 16:47:58.714060   69   
...         ...       ...        ...     ...                        ...  ...   
470888   470889     Tokyo  99.981591    True 2023-06-25 16:47:58.714060   76   
474291   474292    Berlin  99.992728    True 2025-01-02 16:47:58.714060   48   
476522   476523    Sydney  99.986165    True 2023-05-07 16:47:58.714060   23   
479832   479833     Paris  99.987232    True 2024-11-23 16:47:58.714060   48   
485816   485817   Toronto  99.994838   False 2025-10-16 16:47:58.714060   72   

        sessions     revenue  rank  
24

In [20]:
top_10 = duckdb.sql('''SELECT user_id,city from df order by score desc limit 10''')
print(top_10)
%timeit duckdb.sql('''SELECT user_id,city from df order by score desc limit 10''')

┌─────────┬───────────┐
│ user_id │   city    │
│  int32  │  varchar  │
├─────────┼───────────┤
│   83418 │ Seoul     │
│  260671 │ Lima      │
│  412797 │ New York  │
│  324298 │ Toronto   │
│  169257 │ Toronto   │
│   38554 │ New York  │
│  239740 │ Paris     │
│  385192 │ Cape Town │
│  224071 │ Tokyo     │
│    7101 │ Toronto   │
├─────────┴───────────┤
│ 10 rows   2 columns │
└─────────────────────┘

3.19 ms ± 25.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [21]:
total_of_scores = df.sort_values('user_id').assign(running_score=lambda x: x.groupby('city')['score'].cumsum())
print(total_of_scores)
%timeit df.sort_values('user_id').assign(running_score=lambda x: x.groupby('city')['score'].cumsum())

        user_id      city      score  active                signup_date  age  \
0             1     Paris  43.689490   False 2026-02-01 16:47:58.714060   71   
1             2   Toronto  97.403462    True 2025-06-02 16:47:58.714060   39   
2             3  New York  66.148238   False 2025-11-08 16:47:58.714060   23   
3             4      Lima  21.993544   False 2025-05-22 16:47:58.714060   20   
4             5     Paris  91.719953    True 2023-04-02 16:47:58.714060   67   
...         ...       ...        ...     ...                        ...  ...   
499995   499996   Toronto  61.531318   False 2025-04-19 16:47:58.714060   67   
499996   499997    Berlin  74.712924   False 2025-10-25 16:47:58.714060   74   
499997   499998    Sydney  48.430697    True 2023-09-03 16:47:58.714060   27   
499998   499999  New York  89.680757    True 2026-01-24 16:47:58.714060   50   
499999   500000  New York  39.645674    True 2023-12-28 16:47:58.714060   56   

        sessions     revenue  running_s

In [22]:
total_scores_part = duckdb.sql("""SELECT *,SUM(score) OVER (PARTITION BY city ORDER BY user_id) AS running_score FROM df """).df()
print(total_scores_part)
%timeit duckdb.sql("""SELECT *,SUM(score) OVER (PARTITION BY city ORDER BY user_id) AS running_score FROM df """).df()

        user_id    city      score  active                signup_date  age  \
0        490747  Berlin  98.485072   False 2025-08-26 16:47:58.714060   70   
1        490765  Berlin  65.502473    True 2024-05-31 16:47:58.714060   35   
2        490781  Berlin  24.265312    True 2024-05-05 16:47:58.714060   79   
3        490800  Berlin  73.401977    True 2024-08-21 16:47:58.714060   32   
4        490801  Berlin  66.007441    True 2024-02-16 16:47:58.714060   45   
...         ...     ...        ...     ...                        ...  ...   
499995   456313  Sydney  36.534370   False 2025-09-06 16:47:58.714060   65   
499996   456322  Sydney  55.878177    True 2024-01-03 16:47:58.714060   77   
499997   456340  Sydney  52.704199    True 2023-08-03 16:47:58.714060   28   
499998   456348  Sydney  23.856135   False 2024-03-29 16:47:58.714060   35   
499999   456354  Sydney  13.996853   False 2025-01-27 16:47:58.714060   55   

        sessions     revenue  running_score  
0            300 

- In my opinion, duckdb sql is easier to express each query in
- duckdb sql is most of the time faster then pd
- query 2,3,4 have difference the most 

# Task 5

In [23]:
data = {
    "name":       ["Alice", "Bob", "Carol", "David"],
    "age":        [30, 25, 35, 28],
    "salary":     [85_000.0, 72_000.0, 95_000.0, 68_500.0],
    "department": ["Engineering", "Marketing", "Engineering", "Sales"],
    "active":     [True, True, False, True],
}

In [24]:
data_n = pd.DataFrame(data)
data_n

,name,age,salary,department,active
0,Alice,30,85000.0,Engineering,True
1,Bob,25,72000.0,Marketing,True
2,Carol,35,95000.0,Engineering,False
3,David,28,68500.0,Sales,True


In [25]:
import pyarrow as pa
arrow_table = pa.Table.from_pandas(data_n)
arrow_table

pyarrow.Table
name: string
age: int64
salary: double
department: string
active: bool
----
name: [["Alice","Bob","Carol","David"]]
age: [[30,25,35,28]]
salary: [[85000,72000,95000,68500]]
department: [["Engineering","Marketing","Engineering","Sales"]]
active: [[true,true,false,true]]

In [26]:
print(arrow_table.schema)

name: string
age: int64
salary: double
department: string
active: bool
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 810


In [27]:
print(data_n.dtypes)

name           object
age             int64
salary        float64
department     object
active           bool
dtype: object


here only difference just between their names but meaning same

In [28]:
arrow_to_par = pq.write_table(arrow_table, 'employees.parquet')

In [29]:
arrow_table_back = pq.read_table("employees.parquet")

In [30]:
arrow_table_back

pyarrow.Table
name: string
age: int64
salary: double
department: string
active: bool
----
name: [["Alice","Bob","Carol","David"]]
age: [[30,25,35,28]]
salary: [[85000,72000,95000,68500]]
department: [["Engineering","Marketing","Engineering","Sales"]]
active: [[true,true,false,true]]

In [31]:
arrow_to_pandas = arrow_table_back.to_pandas()
arrow_to_pandas

,name,age,salary,department,active
0,Alice,30,85000.0,Engineering,True
1,Bob,25,72000.0,Marketing,True
2,Carol,35,95000.0,Engineering,False
3,David,28,68500.0,Sales,True


In [32]:
arrow_to_pandas.equals(data_n)

True

In [33]:
df_arrow_backed = pd.read_parquet("employees.parquet", dtype_backend="pyarrow")
print(df_arrow_backed.dtypes )

name          string[pyarrow]
age            int64[pyarrow]
salary        double[pyarrow]
department    string[pyarrow]
active          bool[pyarrow]
dtype: object


In [34]:
data_n.dtypes

name           object
age             int64
salary        float64
department     object
active           bool
dtype: object

With dtype_backend="pyarrow", every column has a [pyarrow] suffix.
This means pandas is no longer using NumPy arrays underneath —
instead it keeps the Arrow buffers directly in memory without converting them.

Arrow connects storage (Parquet) with analytical engines (pandas, DuckDB) efficiently, enabling high-performance, memory-efficient, and interoperable data processing.